In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Paramètres par défaut et options de configuration de la transpilation


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Les circuits abstraits doivent être transpilés car les QPU disposent d'un ensemble limité de portes de base et ne peuvent pas exécuter des opérations arbitraires. Le rôle du transpileur est de transformer des circuits arbitraires pour qu'ils puissent s'exécuter sur un QPU donné. Pour cela, il traduit les circuits vers les portes de base prises en charge et introduit des portes SWAP selon les besoins, afin que la connectivité du circuit corresponde à celle du QPU.

Comme expliqué dans [Transpiler avec des gestionnaires de passes](transpile-with-pass-managers), tu peux créer un [gestionnaire de passes](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) à l'aide de la fonction [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) et passer un circuit ou une liste de circuits à sa méthode [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) pour les transpiler. Tu peux appeler `generate_preset_pass_manager` en ne précisant que le niveau d'optimisation et le backend, en laissant les valeurs par défaut pour toutes les autres options, ou bien passer des arguments supplémentaires à la fonction pour affiner la transpilation.

## Utilisation de base sans paramètres
Dans cet exemple, nous passons un circuit et un QPU cible au transpileur sans spécifier d'autres paramètres.

Crée un circuit et visualise le résultat :

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpile le circuit et visualise le résultat :

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Tous les paramètres disponibles
Voici l'ensemble des paramètres disponibles pour la fonction [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). On distingue deux catégories d'arguments : ceux qui décrivent la cible de compilation, et ceux qui influencent le fonctionnement du transpileur.

Tous les paramètres sauf `optimization_level` sont optionnels. Pour plus de détails, consulte la [documentation de l'API Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) — Niveau d'optimisation appliqué aux circuits. Entier compris entre 0 et 3. Plus le niveau est élevé, plus les circuits produits sont optimisés, au prix d'un temps de transpilation plus long. Voir [Définir le niveau d'optimisation du transpileur](set-optimization) pour plus de détails.

### Paramètres décrivant la cible de compilation
Ces arguments décrivent le QPU cible pour l'exécution du circuit, notamment la carte de couplage du QPU (qui décrit la connectivité des qubits), les portes de base prises en charge par le QPU, et les taux d'erreur des portes.

Beaucoup de ces paramètres sont décrits en détail dans [Paramètres couramment utilisés pour la transpilation](common-parameters).

<details>
  <summary>
    **Paramètres du QPU (`Backend`)**
  </summary>

**Paramètres du backend** — Si tu spécifies `backend`, il n'est pas nécessaire de spécifier `target` ni aucune autre option de backend. De même, si tu spécifies `target`, il n'est pas nécessaire de spécifier `backend` ni d'autres options de backend.
- `backend` (Backend) — Si ce paramètre est défini, le transpileur compile le circuit d'entrée pour ce dispositif. Si d'autres options affectant ces paramètres sont définies, comme `coupling_map`, elles ont priorité sur les paramètres du `backend`.
- `target` (Target) — Cible du transpileur pour un backend. Normalement, ce paramètre est spécifié via l'argument `backend`, mais si tu as construit manuellement un objet `Target`, tu peux le fournir ici. Il remplace la cible issue du `backend`.
- `backend_properties` (BackendProperties) — Propriétés renvoyées par un QPU, incluant les erreurs de porte, les erreurs de lecture, les temps de cohérence des qubits, etc. Pour obtenir un QPU fournissant ces informations, exécute `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) — Restriction optionnelle du matériel de contrôle sur la résolution temporelle des instructions. Ces informations sont fournies par la configuration du QPU. Si le QPU n'impose aucune restriction sur l'allocation du temps d'instruction, `timing_constraints` vaut `None` et aucun ajustement n'est effectué. Un QPU peut signaler un ensemble de restrictions, à savoir :
    - `granularity` : Valeur entière représentant la résolution minimale d'une porte pulse en unités de dt. Une porte pulse définie par l'utilisateur doit avoir une durée multiple de cette valeur de granularité.
    - `min_length` : Valeur entière représentant la longueur minimale d'une porte pulse en unités de dt. Une porte pulse définie par l'utilisateur doit être plus longue que cette valeur.
    - `pulse_alignment` : Valeur entière représentant la résolution temporelle du temps de démarrage des instructions de porte. Les instructions de porte doivent commencer à un instant multiple de cette valeur.
    - `acquire_alignment` : Valeur entière représentant la résolution temporelle du temps de démarrage des instructions de mesure. Les instructions de mesure doivent commencer à un instant multiple de cette valeur.
</details>

<details>
  <summary>
    **Paramètres de layout et de topologie**
  </summary>

- `basis_gates` (List[str] | None) — Liste des noms de portes de base vers lesquelles dérouler le circuit. Par exemple `['u1', 'u2', 'u3', 'cx']`. Si `None`, le circuit n'est pas déroulé.
- `coupling_map` (CouplingMap | List[List[int]]) — Carte de couplage dirigée (éventuellement personnalisée) à cibler lors du mapping. Si la carte de couplage est symétrique, les deux directions doivent être spécifiées. Les formats suivants sont pris en charge :
    - Instance de CouplingMap
    - Liste — doit être fournie sous forme de matrice d'adjacence, où chaque entrée spécifie toutes les interactions bidirectionnelles à deux qubits prises en charge par le QPU. Par exemple : [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) — Correspondance entre les opérations du circuit et les planifications pulse. Si `None`, l'`instruction_schedule_map` du QPU est utilisé.
</details>

### Paramètres influençant le fonctionnement du transpileur
Ces paramètres ont un impact sur des étapes spécifiques de la transpilation. Certains peuvent affecter plusieurs étapes, mais n'ont été listés que sous une seule pour simplifier. Si tu spécifies un argument, comme `initial_layout` pour les qubits que tu souhaites utiliser, cette valeur remplace toutes les passes susceptibles de le modifier. En d'autres termes, le transpileur ne changera rien à ce que tu spécifies manuellement. Pour des détails sur les étapes spécifiques, voir [Étapes du transpileur](transpiler-stages).

<details>
  <summary>
    **Étape d'initialisation**
  </summary>

- `hls_config` (HLSConfig) — Classe de configuration optionnelle `HLSConfig` passée directement à la passe de transformation `HighLevelSynthesis`. Cette classe de configuration te permet de spécifier les listes d'algorithmes de synthèse et leurs paramètres pour divers objets de haut niveau.
- `init_method` (str) — Nom du plugin à utiliser pour l'étape d'initialisation. Par défaut, aucun plugin externe n'est utilisé. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `init` comme argument de nom d'étape.
- `unitary_synthesis_method` (str) — Nom de la méthode de synthèse unitaire à utiliser. Par défaut, `default` est utilisé. Tu peux voir la liste des plugins installés en exécutant `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) — Dictionnaire de configuration optionnel passé directement au plugin de synthèse unitaire. Par défaut, ce paramètre n'a aucun effet car la méthode de synthèse unitaire par défaut n'accepte pas de configuration personnalisée. Une configuration personnalisée n'est nécessaire que lorsqu'un plugin de synthèse unitaire est spécifié avec l'argument `unitary_synthesis`. Comme cela est propre à chaque plugin, consulte la documentation du plugin pour savoir comment utiliser cette option.
</details>

<details>
  <summary>
    **Étape de layout**
  </summary>

- `initial_layout` (Layout | Dict | List) — Position initiale des qubits virtuels sur les qubits physiques. Si ce layout rend le circuit compatible avec les contraintes de `coupling_map`, il sera utilisé. Le layout final n'est pas garanti d'être identique, car le transpileur peut permuter les qubits via des SWAP ou d'autres moyens. Pour plus de détails, voir la [section Initial layout.](common-parameters#initial-layout)
- `layout_method` (str) — Nom de la passe de sélection de layout (`default`, `dense`, `sabre` et `trivial`). Il peut également s'agir du nom d'un plugin externe à utiliser pour l'étape de layout. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `layout` comme argument `stage_name`. La valeur par défaut est `sabre`.
</details>

<details>
  <summary>
    **Étape de routage**
  </summary>

- `routing_method` (str) — Nom de la passe de routage (`basic`, `lookahead`, `default`, `sabre` ou `none`). Il peut également s'agir du nom d'un plugin externe à utiliser pour l'étape de routage. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `routing` comme argument `stage_name`. La valeur par défaut est `sabre`.
</details>

<details>
  <summary>
    **Étape de traduction**
  </summary>

- `translation_method` (str) — Nom de la passe de traduction (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Il peut également s'agir du nom d'un plugin externe à utiliser pour l'étape de traduction. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `translation` comme argument `stage_name`. La valeur par défaut est `translator`.
</details>

<details>
  <summary>
    **Étape d'optimisation**
  </summary>

- `approximation_degree` (float, compris entre 0 et 1 | None) — Curseur heuristique utilisé pour l'approximation du circuit (1,0 = aucune approximation, 0,0 = approximation maximale). La valeur par défaut est 1,0. Spécifier `None` fixe le degré d'approximation au taux d'erreur rapporté. Voir la [section Degré d'approximation](common-parameters#approx-degree) pour plus de détails.
- `optimization_method` (str) — Nom du plugin à utiliser pour l'étape d'optimisation. Par défaut, aucun plugin externe n'est utilisé. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `optimization` comme argument `stage_name`.
</details>

<details>
  <summary>
    **Étape de planification**
  </summary>

- `scheduling_method` (str) — Nom de la passe de planification. Il peut également s'agir du nom d'un plugin externe à utiliser pour l'étape de planification. Tu peux voir la liste des plugins installés en exécutant `list_stage_plugins()` avec `scheduling` comme argument `stage_name`.
  * `'as_soon_as_possible'` : Planifie les instructions de manière gourmande, le plus tôt possible sur une ressource qubit (alias : `asap`).
  * `'as_late_as_possible'` : Planifie les instructions le plus tard possible, c'est-à-dire en maintenant les qubits dans l'état fondamental autant que possible (alias : `alap`). C'est la valeur par défaut.
</details>

<details>
  <summary>
    **Exécution du transpileur**
  </summary>

- `seed_transpiler` (int) — Définit les graines aléatoires pour les parties stochastiques du transpileur.
</details>

Les valeurs par défaut suivantes sont utilisées si tu ne spécifies aucun des paramètres ci-dessus. Consulte la [page de référence de l'API](../api/qiskit/transpiler_preset) de la méthode pour plus d'informations :

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>